In [1]:
import pandas as pd
import numpy as nm

In [2]:
import pandas as pd
import numpy as np
import os
import sys
# import matplotlib.pyplot as plt
from pathlib import Path

# import sys, subprocess, importlib
print('Notebook Python executable:', sys.executable) # to check the python environment and the path from the local system
# importlib.import_module('matplotlib')
# import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

Notebook Python executable: c:\Users\HimanshuSomvanshi\Documents\Vienna Course\.venv\Scripts\python.exe


In [3]:
# Resolve CSV path robustly for notebook runs from different working directories
candidate_paths = [
    Path("sales_data.csv"),
    Path("data") / "raw data" / "sales_data.csv",
    Path.cwd() / "data" / "raw data" / "sales_data.csv",
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    checked = "\n".join(str(p) for p in candidate_paths)
    raise FileNotFoundError(f"Could not find sales_data.csv. Checked:\n{checked}")

# Read CSV using first row as header
df_sales = pd.read_csv(
    csv_path,
    encoding="ISO-8859-1",
    low_memory=False,
    header=0,
    skipinitialspace=True,
    on_bad_lines="skip",
)

# Normalize column names: lowercase + snake_case
df_sales.columns = (
    df_sales.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

# Rename seasonality column to season
df_sales = df_sales.rename(columns={"seasonality": "season"})

# Convert requested columns to target dtypes
string_cols = [
    "category",
    "product_id",
    "region",
    "season",
    "store_id",
    "weather_condition",
]
for col in string_cols:
    if col in df_sales.columns:
        df_sales[col] = df_sales[col].astype("string")

if "date" in df_sales.columns:
    df_sales["date"] = pd.to_datetime(df_sales["date"], errors="coerce")

print("\n" + "=" * 60)
print("SALES TABLE - Initial Assessment")
print("=" * 60)
print(df_sales.head(5))

# print("\nNormalized column names:")
# print(df_sales.columns.tolist())


SALES TABLE - Initial Assessment
        date store_id product_id     category region  inventory_level  \
0 2022-01-01     S001      P0001  Electronics  North              195   
1 2022-01-01     S001      P0002     Clothing  North              117   
2 2022-01-01     S001      P0003     Clothing  North              247   
3 2022-01-01     S001      P0004  Electronics  North              139   
4 2022-01-01     S001      P0005    Groceries  North              152   

   units_sold  units_ordered  price  discount weather_condition  promotion  \
0         102            252  72.72         5             Snowy          0   
1         117            249  80.16        15             Snowy          1   
2         114            612  62.94        10             Snowy          1   
3          45            102  87.63        10             Snowy          0   
4          65            271  54.41         0             Snowy          0   

   competitor_pricing  season  epidemic  demand  
0       

In [4]:
print(df_sales.describe())

                      date  inventory_level    units_sold  units_ordered  \
count                76000     76000.000000  76000.000000   76000.000000   
mean   2023-01-15 12:00:00       301.062842     88.827316      89.090645   
min    2022-01-01 00:00:00         0.000000      0.000000       0.000000   
25%    2022-07-09 18:00:00       136.000000     58.000000       0.000000   
50%    2023-01-15 12:00:00       227.000000     84.000000       0.000000   
75%    2023-07-24 06:00:00       408.000000    114.000000     121.000000   
max    2024-01-30 00:00:00      2267.000000    426.000000    1616.000000   
std                    NaN       226.510161     43.994525     162.404627   

              price      discount     promotion  competitor_pricing  \
count  76000.000000  76000.000000  76000.000000        76000.000000   
mean      67.726028      9.087039      0.328947           69.454029   
min        4.740000      0.000000      0.000000            4.290000   
25%       31.997500      5.0000

In [5]:
pick_missing_values_count = df_sales.isnull().sum()
print("Missing values in df_sales:\n")
print(pick_missing_values_count,"\n")

Missing values in df_sales:

date                  0
store_id              0
product_id            0
category              0
region                0
inventory_level       0
units_sold            0
units_ordered         0
price                 0
discount              0
weather_condition     0
promotion             0
competitor_pricing    0
season                0
epidemic              0
demand                0
dtype: int64 



In [6]:
df_sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76000 entries, 0 to 75999
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   date                76000 non-null  datetime64[ns]
 1   store_id            76000 non-null  string        
 2   product_id          76000 non-null  string        
 3   category            76000 non-null  string        
 4   region              76000 non-null  string        
 5   inventory_level     76000 non-null  int64         
 6   units_sold          76000 non-null  int64         
 7   units_ordered       76000 non-null  int64         
 8   price               76000 non-null  float64       
 9   discount            76000 non-null  int64         
 10  weather_condition   76000 non-null  string        
 11  promotion           76000 non-null  int64         
 12  competitor_pricing  76000 non-null  float64       
 13  season              76000 non-null  string    

In [7]:
# Validate text-like columns and build dtype recommendations
import re

text_cols = df_sales.select_dtypes(include=["object", "string"]).columns.tolist()

if not text_cols:
    print("No object/string columns found.")
else:
    summary_rows = []
    alpha_pattern = re.compile(r"[A-Za-z]")
    total_rows = len(df_sales)

    for col in text_cols:
        s = df_sales[col].dropna().astype(str).str.strip()

        if s.empty:
            summary_rows.append({
                "column": col,
                "current_dtype": str(df_sales[col].dtype),
                "contains_alphabets": False,
                "alpha_ratio_%": 0.0,
                "numeric_like_ratio_%": 0.0,
                "n_unique": 0,
                "unique_ratio_%": 0.0,
                "recommended_dtype": "string",
                "recommendation_note": "empty/non-informative values",
            })
            continue

        has_alpha = s.str.contains(alpha_pattern, regex=True, na=False)
        alpha_ratio = has_alpha.mean() * 100

        numeric_like_ratio = pd.to_numeric(s, errors="coerce").notna().mean() * 100
        n_unique = s.nunique(dropna=True)
        unique_ratio = (n_unique / total_rows) * 100 if total_rows else 0.0

        # Explicit rule requested: season should be treated as categorical candidate
        if col == "season":
            recommended_dtype = "category"
            note = "categorical candidate (explicit business label set)"
        elif numeric_like_ratio == 100 and alpha_ratio == 0:
            recommended_dtype = "numeric (int/float)"
            note = "all values parse as numeric"
        elif alpha_ratio > 0 and unique_ratio <= 20:
            recommended_dtype = "category"
            note = "low-cardinality text column"
        else:
            recommended_dtype = "string"
            note = "high-cardinality or mixed text"

        summary_rows.append({
            "column": col,
            "current_dtype": str(df_sales[col].dtype),
            "contains_alphabets": bool(has_alpha.any()),
            "alpha_ratio_%": round(alpha_ratio, 2),
            "numeric_like_ratio_%": round(numeric_like_ratio, 2),
            "n_unique": int(n_unique),
            "unique_ratio_%": round(unique_ratio, 2),
            "recommended_dtype": recommended_dtype,
            "recommendation_note": note,
        })

    dtype_recommendations = (
        pd.DataFrame(summary_rows)
        .sort_values("column")
        .reset_index(drop=True)
    )

    print("Final dtype recommendations table:")
    display(dtype_recommendations)

    print("\nColumns explicitly marked as categorical candidate:")
    print(
        dtype_recommendations.loc[
            dtype_recommendations["recommended_dtype"].eq("category"), "column"
        ].tolist()
    )

Final dtype recommendations table:


,column,current_dtype,contains_alphabets,alpha_ratio_%,numeric_like_ratio_%,n_unique,unique_ratio_%,recommended_dtype,recommendation_note
0,category,string,True,100.0,0.0,5,0.01,category,low-cardinality text column
1,product_id,string,True,100.0,0.0,20,0.03,category,low-cardinality text column
2,region,string,True,100.0,0.0,4,0.01,category,low-cardinality text column
3,season,string,True,100.0,0.0,4,0.01,category,categorical candidate (explicit business label...
4,store_id,string,True,100.0,0.0,5,0.01,category,low-cardinality text column
5,weather_condition,string,True,100.0,0.0,4,0.01,category,low-cardinality text column



Columns explicitly marked as categorical candidate:
['category', 'product_id', 'region', 'season', 'store_id', 'weather_condition']


In [8]:
duplicates = df_sales.duplicated()

print(f"Total duplicate rows: {duplicates.sum()}")
print(f"Total rows: {len(df_sales)}")
print(f"Percentage: {(duplicates.sum()/len(df_sales))*100:.2f}%")

Total duplicate rows: 0
Total rows: 76000
Percentage: 0.00%


In [9]:
df_sales.head()

,date,store_id,product_id,category,region,inventory_level,units_sold,units_ordered,price,discount,weather_condition,promotion,competitor_pricing,season,epidemic,demand
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157
3,2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,Snowy,0,85.19,Winter,0,52
4,2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,Snowy,0,51.63,Winter,0,59


In [10]:
# Save processed dataset to data/processed data
from pathlib import Path

processed_dir = Path("data") / "processed data"
processed_dir.mkdir(parents=True, exist_ok=True)

output_file = processed_dir / "sales_data_processed.csv"
df_sales.to_csv(output_file, index=False)

print(f"Processed file saved to: {output_file.resolve()}")
print(f"Rows: {len(df_sales)}, Columns: {df_sales.shape[1]}")

Processed file saved to: C:\Users\HimanshuSomvanshi\Documents\3rd Sem_technology_management\data\processed data\sales_data_processed.csv
Rows: 76000, Columns: 16
